# 3DShape2VecSet on Kaggle: autoencoder training from a config file

This notebook clones the repository, reads a training config from it, and runs the first stage (`main_ae.py`) with `torchrun`. Two configs are provided, both with **C₀ = 2** latent channels:

- `configs/kl_d512_m256_l2_d8_2cat.yaml` (default): a reduced run sized for Kaggle. It uses 256 latents, an 8-block decoder, the airplane and chair categories only, mixed precision, and 40 epochs.
- `configs/kl_d512_m512_l2.yaml`: the README's full setup. It needs thousands of GPU-hours, which is far beyond Kaggle's quota.

**Two modes**, set `MODE` in the first code cell:
- `"smoke"`: runs `kaggle/smoke_test.py`, a GPU smoke test of about 15–25 minutes. It checks fp16, both GPUs, stopping and resuming, memory, `eval.py`, and that the model learns simple synthetic shapes. Then it prints the time estimate. It doesn't train, and it works without the ShapeNet dataset attached. The report is saved to `/kaggle/working/smoke_test_report.json`.
- `"train"`: trains with the config.

Run `"smoke"` once in an interactive session first, then `"train"` with *Save & Run All*. Full instructions are in `kaggle/KAGGLE.md`.

**Before you run it**
1. **Accelerator:** *GPU T4 x2*. **Internet:** on (needed for `git clone` and `pip`).
2. **Data:** upload the preprocessed ShapeNet data (`occupancies.zip` and `surfaces.zip` from the README's link) as a private Kaggle Dataset and add it as an input. The notebook searches `/kaggle/input` for the `ShapeNetV2_point` and `ShapeNetV2_watertight` folders, so the dataset's folder layout doesn't matter.
3. **Run it with "Save Version → Save & Run All"**, so it keeps running after you close the tab and `/kaggle/working` is saved as the version's output.

**Continuing in a later session:** add this notebook's previous version output as an input. With `resume: auto`, the notebook picks up the newest checkpoint it finds there, including a mid-epoch one. Training saves a rolling mid-epoch checkpoint every `save_every_steps` optimizer steps, and saves again and stops cleanly when `time_budget_hours` runs out, so a session loses at most a few minutes of work. Keep the same GPU count and batch size between sessions, so the resumed epoch continues in the same data order.

**Time:** a full run takes far longer than one session. The benchmark cell measures real speed and prints an estimate before training starts. Read the warning in that cell's section.

In [ ]:
# ---- Settings ----
MODE = "smoke"                               # "smoke": GPU smoke test + time estimate, no training. "train": train.
REPO_URL = "https://github.com/ugwnate-hash/3DShape2VecSet.git"
BRANCH = "fix/review-issues"                 # has the PyTorch 2.x fixes; upstream master does not start on current PyTorch
CONFIG = "configs/kl_d512_m256_l2_d8_2cat.yaml"   # path inside the repository

RUN_DIR = "/tmp/run"                         # code, shims and data links (not saved)
REPO_DIR = f"{RUN_DIR}/3DShape2VecSet"
SHIM_DIR = f"{RUN_DIR}/shims"
WORK = "/kaggle/working"                     # saved as the version's output

## 1. Clone the repository and read the config

In [ ]:
import os, sys, subprocess, shutil, glob, json, re, time, signal, threading

os.makedirs(RUN_DIR, exist_ok=True)
shutil.rmtree(REPO_DIR, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
print("commit", COMMIT)

import yaml
with open(os.path.join(REPO_DIR, CONFIG)) as f:
    cfg = yaml.safe_load(f)

MODEL = cfg["model"]
HW = cfg["hardware"]
KG = cfg["kaggle"]
NPROC = int(HW["nproc_per_node"])
EFF_BATCH = NPROC * HW["batch_size"] * HW["accum_iter"]
OUT_DIR = f"{WORK}/output/ae/{MODEL}"
os.makedirs(OUT_DIR, exist_ok=True)

print(yaml.safe_dump(cfg, sort_keys=False))
print("effective batch size:", EFF_BATCH)
if EFF_BATCH != 512:
    print("WARNING: the README's effective batch size is 512. This run uses", EFF_BATCH,
          "so its learning rate and number of optimizer steps differ.")

## 2. Check the GPUs and install dependencies

In [ ]:
gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()
print("\n".join(gpus) or "no GPU found")
if len(gpus) < NPROC:
    raise RuntimeError(f"The config asks for {NPROC} GPUs but {len(gpus)} are available. "
                       "Set the accelerator to 'GPU T4 x2', or change hardware.nproc_per_node "
                       "(and accum_iter, to keep the effective batch size).")

def pip(*args, check=True):
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=check)

pip("einops", "timm", "tensorboard", "h5py", "pyyaml", "PyMCubes", "trimesh", "scipy")   # last three: eval.py

In [ ]:
# torch_cluster provides fps (farthest point sampling) in models_ae.py.
# Try a prebuilt wheel that matches Kaggle's PyTorch and CUDA. If there is none,
# fall back to a pure-PyTorch fps with the same interface (a few percent slower).
import torch

def torch_cluster_works(env=None):
    r = subprocess.run([sys.executable, "-c", "from torch_cluster import fps"], capture_output=True, text=True, env=env)
    return r.returncode == 0

major_minor = ".".join(torch.__version__.split("+")[0].split(".")[:2])
cuda = (torch.version.cuda or "").replace(".", "")
wheel_index = f"https://data.pyg.org/whl/torch-{major_minor}.0+cu{cuda}.html"
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| wheel index", wheel_index)

USE_SHIM = False
if not torch_cluster_works():
    pip("--only-binary", ":all:", "torch-cluster", "-f", wheel_index, check=False)
    if not torch_cluster_works():
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torch-cluster"])
        USE_SHIM = True

if USE_SHIM:
    os.makedirs(f"{SHIM_DIR}/torch_cluster", exist_ok=True)
    with open(f"{SHIM_DIR}/torch_cluster/__init__.py", "w") as f:
        f.write("""
import torch

def fps(src, batch=None, ratio=0.5, random_start=True):
    # Same output as torch_cluster.fps when every batch element has the same number of points,
    # which is always the case in models_ae.py: indices into src, grouped by batch element.
    if batch is None:
        batch = torch.zeros(src.shape[0], dtype=torch.long, device=src.device)
    B = int(batch.max()) + 1
    N = src.shape[0] // B
    M = int(-(-ratio * N // 1))
    p = src.view(B, N, -1)
    ar = torch.arange(B, device=src.device)
    idx = torch.empty(B, M, dtype=torch.long, device=src.device)
    dist = torch.full((B, N), float("inf"), device=src.device, dtype=src.dtype)
    cur = (torch.randint(0, N, (B,), device=src.device) if random_start
           else torch.zeros(B, dtype=torch.long, device=src.device))
    for i in range(M):
        idx[:, i] = cur
        dist = torch.minimum(dist, ((p - p[ar, cur][:, None]) ** 2).sum(-1))
        cur = dist.argmax(-1)
    return (idx + ar[:, None] * N).view(-1)
""")
    print("No torch_cluster wheel for this PyTorch build: using the pure-PyTorch fps fallback.")
else:
    print("torch_cluster OK")

ENV = dict(os.environ)
ENV["PYTHONPATH"] = os.pathsep.join(p for p in [SHIM_DIR if USE_SHIM else "", REPO_DIR, ENV.get("PYTHONPATH", "")] if p)

## 3. Find the data

In [ ]:
def find_dir(name, root="/kaggle/input", max_depth=5):
    root_depth = root.rstrip("/").count("/")
    for cur, dirs, _ in os.walk(root, followlinks=True):
        if name in dirs:
            return os.path.join(cur, name)
        if cur.count("/") - root_depth >= max_depth:
            dirs[:] = []
        else:
            dirs[:] = [d for d in dirs if not re.fullmatch(r"\d{8}", d)]  # don't descend into category folders
    return None

DATA = None
if KG["data_root"] == "auto":
    point_dir, mesh_dir = find_dir("ShapeNetV2_point"), find_dir("ShapeNetV2_watertight")
    if not point_dir or not mesh_dir:
        msg = ("Could not find ShapeNetV2_point and ShapeNetV2_watertight under /kaggle/input. "
               "Add the dataset as an input, or set kaggle.data_root in the config.")
        if MODE == "train":
            raise FileNotFoundError(msg)
        print(msg, "\nSmoke mode: continuing with synthetic data only (the real-data check and the time estimate are skipped).")
if KG["data_root"] == "auto" and DATA is None and point_dir and mesh_dir:
    DATA = f"{RUN_DIR}/data"
    os.makedirs(DATA, exist_ok=True)
    for src, name in [(point_dir, "ShapeNetV2_point"), (mesh_dir, "ShapeNetV2_watertight")]:
        link = os.path.join(DATA, name)
        if os.path.islink(link):
            os.remove(link)
        os.symlink(src, link)
elif KG["data_root"] != "auto":
    DATA = KG["data_root"]

CATEGORIES = [c.strip() for c in str(cfg["args"].get("categories") or "").split(",") if c.strip()]
missing = [c for c in CATEGORIES if DATA and not os.path.isdir(f"{DATA}/ShapeNetV2_point/{c}")]
if missing:
    raise FileNotFoundError(f"categories not found in the data: {missing}")

def count_split(split):
    n = 0
    for lst in glob.glob(f"{DATA}/ShapeNetV2_point/*/{split}.lst"):
        if CATEGORIES and os.path.basename(os.path.dirname(lst)) not in CATEGORIES:
            continue
        with open(lst) as f:
            n += sum(1 for line in f if line.strip())
    return n

REPLICA = 16  # util/shapenet.py: each training shape appears 16 times per epoch
N_TRAIN = N_VAL = None
if DATA:
    N_TRAIN, N_VAL = count_split("train"), count_split("val")
    print("data:", DATA, "| categories:", CATEGORIES or "all")
    print(f"train shapes: {N_TRAIN}  val shapes: {N_VAL}  samples per epoch: {N_TRAIN * REPLICA}")
    assert N_TRAIN > 0, "no train.lst files found"

## 4. Find a checkpoint to resume from

In [ ]:
def ckpt_position(path):
    """(epoch, iteration) that training continues from after loading this checkpoint."""
    name = os.path.basename(path)
    m = re.fullmatch(r"checkpoint-(\d+)-it(\d+)\.pth", name)   # mid-epoch
    if m:
        return (int(m.group(1)), int(m.group(2)))
    m = re.fullmatch(r"checkpoint-(\d+)\.pth", name)            # end of epoch
    return (int(m.group(1)) + 1, 0) if m else (-1, -1)

RESUME = None
if KG["resume"] == "auto":
    found = [p for root in ("/kaggle/input", WORK)
             for p in glob.glob(f"{root}/**/{MODEL}/checkpoint-*.pth", recursive=True)]
    found = [p for p in found if ckpt_position(p) >= (0, 0)]
    if found:
        RESUME = max(found, key=ckpt_position)
elif KG["resume"] not in (None, "none"):
    RESUME = KG["resume"]

START_EPOCH, START_ITER = ckpt_position(RESUME) if RESUME else (0, 0)
if RESUME:
    print(f"Resuming from {RESUME} (continues at epoch {START_EPOCH}, iteration {START_ITER})")
    # keep the epoch log continuous across sessions
    prev_log = os.path.join(os.path.dirname(RESUME), "log.txt")
    if os.path.exists(prev_log) and os.path.dirname(RESUME) != OUT_DIR:
        shutil.copy(prev_log, os.path.join(OUT_DIR, "log.txt"))
else:
    print("No checkpoint: starting from epoch 0")

## 5. Smoke test (MODE = "smoke" only)

Runs `kaggle/smoke_test.py`. Each check prints PASS, FAIL or SKIP. The full report, with every measured number, is written to `/kaggle/working/smoke_test_report.json`. If something fails, paste that file back.

In [ ]:
SMOKE_REPORT = f"{WORK}/smoke_test_report.json"
if MODE == "smoke":
    smoke_cmd = [sys.executable, "-u", "kaggle/smoke_test.py", "--config", CONFIG, "--report", SMOKE_REPORT,
                 "--workdir", f"{RUN_DIR}/smoke"]
    if DATA:
        smoke_cmd += ["--data", DATA]
    proc = subprocess.Popen(smoke_cmd, cwd=REPO_DIR, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        if line.startswith(("[PASS]", "[FAIL]", "[SKIP]", "SUMMARY", "model ", "report ")):
            print(line, end="")
    proc.wait()
    print("smoke test exit code:", proc.returncode, "(0 = all checks passed)")
else:
    print("MODE is", MODE, "- smoke test skipped")

## 6. Benchmark and time estimate

Two measurements, then a projection for the whole run:
- **GPU:** a few training steps on one GPU, with the config's batch size and mixed-precision setting, on random inputs.
- **Data loading:** one minute of reading real training samples with as many workers as the run will use. Kaggle has only 4 CPU cores, so with a small, fast model the data loader can be the bottleneck.

The projection uses whichever is slower. Validation (every 5 epochs) is not included.

In [ ]:
BENCH = r"""
import sys, time, json, torch
cfg = json.loads(sys.argv[1])
import models_ae
dev = "cuda:0"
bs, steps, pc = cfg["batch_size"], cfg["steps"], cfg["point_cloud_size"]
torch.backends.cudnn.benchmark = True
model = models_ae.__dict__[cfg["model"]](N=pc).to(dev).train()
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.cuda.amp.GradScaler(enabled=cfg["amp"])
crit = torch.nn.BCEWithLogitsLoss()
def step():
    surface = torch.rand(bs, pc, 3, device=dev) * 2 - 1
    points = torch.rand(bs, 2048, 3, device=dev) * 2 - 1   # 1024 volume + 1024 near-surface queries
    labels = (torch.rand(bs, 2048, device=dev) > 0.5).float()
    with torch.autocast("cuda", dtype=torch.float16, enabled=cfg["amp"]):
        out = model(surface, points)
        loss = crit(out["logits"].float(), labels) + 1e-3 * out["kl"].float().sum() / bs
    scaler.scale(loss).backward()
    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
try:
    for _ in range(3):
        step()
    torch.cuda.synchronize(); t = time.time()
    for _ in range(steps):
        step()
    torch.cuda.synchronize()
    sec = (time.time() - t) / steps
    print(json.dumps({"sec_per_step": sec, "peak_gb": torch.cuda.max_memory_allocated() / 1e9}))
except torch.cuda.OutOfMemoryError:
    print(json.dumps({"oom": True}))
"""

DATA_BENCH = r"""
import sys, time, json, argparse, torch
cfg = json.loads(sys.argv[1])
from util.datasets import build_shape_surface_occupancy_dataset
args = argparse.Namespace(data_path=cfg["data"], point_cloud_size=cfg["point_cloud_size"], categories=cfg["categories"])
ds = build_shape_surface_occupancy_dataset("train", args)
dl = torch.utils.data.DataLoader(ds, batch_size=cfg["batch_size"], shuffle=True,
                                 num_workers=cfg["workers"], drop_last=True)
def batches():   # loop over the data, so a small category subset can't run out mid-test
    while True:
        yield from dl
it = batches()
next(it)   # worker start-up
t, n = time.time(), 0
while time.time() - t < cfg["seconds"]:
    next(it); n += cfg["batch_size"]
print(json.dumps({"samples_per_s": n / (time.time() - t)}))
"""

EST = None
if KG["benchmark_steps"] > 0 and not DATA:
    print("No real data: benchmark and time estimate skipped.")
elif KG["benchmark_steps"] > 0:
    bench_cfg = {"model": MODEL, "batch_size": HW["batch_size"], "steps": KG["benchmark_steps"],
                 "point_cloud_size": cfg["args"]["point_cloud_size"], "amp": bool(cfg["args"].get("amp"))}
    r = subprocess.run([sys.executable, "-c", BENCH, json.dumps(bench_cfg)], capture_output=True, text=True, env=ENV)
    lines = [l for l in r.stdout.splitlines() if l.startswith("{")]
    if r.returncode != 0 or not lines:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError("benchmark failed")
    res = json.loads(lines[-1])
    if res.get("oom"):
        raise RuntimeError(f"batch_size {HW['batch_size']} does not fit in GPU memory. Lower hardware.batch_size "
                           "and raise accum_iter to keep the effective batch size.")
    DDP_EFFICIENCY = 0.9  # assumed: gradient all-reduce between the two T4s
    gpu_sps = HW["batch_size"] / res["sec_per_step"] * NPROC * DDP_EFFICIENCY

    data_cfg = {"data": DATA, "point_cloud_size": cfg["args"]["point_cloud_size"],
                "categories": cfg["args"].get("categories"), "batch_size": HW["batch_size"],
                "workers": HW["num_workers"] * NPROC, "seconds": 60}
    r = subprocess.run([sys.executable, "-c", DATA_BENCH, json.dumps(data_cfg)], capture_output=True, text=True,
                       env=ENV, cwd=REPO_DIR)
    lines = [l for l in r.stdout.splitlines() if l.startswith("{")]
    if r.returncode != 0 or not lines:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError("data-loading benchmark failed")
    data_sps = json.loads(lines[-1])["samples_per_s"]
    sps = min(gpu_sps, data_sps)
    print(f"GPU throughput ({NPROC} GPUs): about {gpu_sps:.0f} samples/s | data loading: about {data_sps:.0f} samples/s"
          f" -> limited by {'data loading' if data_sps < gpu_sps else 'the GPUs'}")
    if data_sps < 0.8 * gpu_sps:
        print("  The GPUs will wait for data. The data benchmark reads from a cold disk cache, so later epochs may be "
              "faster once the files are cached in RAM.")
    epoch_h = N_TRAIN * REPLICA / sps / 3600
    iters_per_epoch = N_TRAIN * REPLICA / (NPROC * HW["batch_size"])
    epochs_left = cfg["args"]["epochs"] - START_EPOCH - START_ITER / iters_per_epoch
    total_h = epoch_h * epochs_left
    EST = {"epoch_hours": epoch_h, "total_hours": total_h}
    print(f"one GPU: {res['sec_per_step']:.2f} s per step of {HW['batch_size']} | peak memory {res['peak_gb']:.1f} GB")
    print(f"training throughput: about {sps:.0f} samples/s")
    print(f"one epoch ({N_TRAIN * REPLICA} samples): about {epoch_h:.2f} h")
    print(f"{epochs_left:.2f} remaining epochs: about {total_h:.1f} GPU-session hours "
          f"= {total_h / KG['time_budget_hours']:.1f} sessions of {KG['time_budget_hours']} h "
          f"= {total_h / 30:.1f} weeks of Kaggle's 30 h/week GPU quota")
    print(f"this session ({KG['time_budget_hours']} h): about {KG['time_budget_hours'] / epoch_h:.2f} epochs")
    if cfg["args"].get("save_every_steps"):
        print(f"mid-epoch checkpoint every {cfg['args']['save_every_steps']} optimizer steps = about "
              f"{cfg['args']['save_every_steps'] * EFF_BATCH / sps / 60:.0f} min")

## 7. Train

In [ ]:
def flags(d):
    out = []
    for k, v in d.items():
        if v is None or v is False or k == "nproc_per_node":
            continue
        out += [f"--{k}"] if v is True else [f"--{k}", str(v)]
    return out

cmd = ["torchrun", "--standalone", f"--nproc_per_node={NPROC}", "main_ae.py",
       "--model", MODEL, "--data_path", str(DATA), "--output_dir", OUT_DIR, "--log_dir", OUT_DIR,
       *flags(cfg["args"]), *flags(HW), "--max_hours", str(KG["time_budget_hours"])]
if RESUME:
    cmd += ["--resume", RESUME]

with open(f"{OUT_DIR}/run_info.json", "w") as f:
    json.dump({"repo": REPO_URL, "branch": BRANCH, "commit": COMMIT, "config": cfg,
               "command": cmd, "resume": RESUME, "estimate": EST, "torch_cluster_shim": USE_SHIM,
               "started": time.strftime("%Y-%m-%d %H:%M:%S")}, f, indent=2)
print(" ".join(map(str, cmd)))

if MODE != "train":
    print(f'MODE is "{MODE}": not training. Set MODE = "train" in the first cell to train.')
elif not KG["train"]:
    print("kaggle.train is false: not training.")
else:
    budget_s = (KG["time_budget_hours"] + 0.5) * 3600   # safety net: training should already have stopped itself
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, start_new_session=True)
    def stop():
        print(f"\nTraining still running {KG['time_budget_hours'] + 0.5} h after launch: killing it.", flush=True)
        try:
            os.killpg(proc.pid, signal.SIGINT)
            proc.wait(timeout=120)
        except Exception:
            os.killpg(proc.pid, signal.SIGKILL)
    timer = threading.Timer(budget_s, stop)
    timer.start()
    with open(f"{OUT_DIR}/stdout.log", "a") as log:
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
    proc.wait()
    timer.cancel()
    print("torchrun exit code:", proc.returncode)

## 8. Tidy up the output

In [ ]:
# Keep only the newest checkpoints, so the output stays under Kaggle's 20 GB limit.
ckpts = sorted(glob.glob(f"{OUT_DIR}/checkpoint-*.pth"), key=ckpt_position)
for p in ckpts[:max(0, len(ckpts) - KG["keep_checkpoints"])]:
    os.remove(p)
print("checkpoints kept:", [os.path.basename(p) for p in sorted(glob.glob(f"{OUT_DIR}/checkpoint-*.pth"), key=ckpt_position)] or "none")

log_txt = f"{OUT_DIR}/log.txt"
if os.path.exists(log_txt):
    with open(log_txt) as f:
        rows = [json.loads(l) for l in f if l.strip()]
    for r in rows[-5:]:
        print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()
               if k in ("epoch", "train_loss", "train_iou", "train_loss_kl", "test_iou")})